# Automated promoter library redesign

這本 notebook 從 high-throughput PKL database 選取元件，不進行 de novo sequence generation。

每個 element 的 pooling 邊界由 `CONFIG.mutable_energy_fraction_ranges` 逐元素指定，形式是
`(lower_fraction, upper_fraction, n_bins)` —— 在該元素自己的觀測 min–max energy 軸上，
要納入哪一段、切成幾個等寬 bin。**每個 bin 各出一個 mutable 版本**，所以 N 個 bin 給出
`v1..vN`；`CONFIG.consensus` 每個 element 可以放**一條或多條** locked 序列，由弱到強依序填在
`v{N+1}` 之後的槽位，且 `v1..vN` 必須全部弱於**最弱的那一條** locked。

目前 UP 鎖了兩條（`v4 = TGGACTGATATATACAAAA`、`v5 = TAAAAAATTTGGAAAATAG`）因此只切 3 個 bin，
其餘元素各鎖一條、切 4 個 bin —— 每個 element 都仍是 5 個版本，所以文庫維持 `5^6 = 15,625`。

Spacer 是例外：`Spacer_v2` 來自 bin 2，`Spacer_v3 = Spacer_v2[:-2] + "TG"`（衍生而非取自 bin 3），
兩者是同一個 coupled design unit，所以 spacer 至少要 3 個 bin。

每個 design state 組合成 `∏(N_e + L_e)` variants（`L_e` = 該元素 locked 條數）；六個元素都是 5 個版本時就是
`5^6 = 15,625`（Batch 0 會把實際數字印出來）。使用固定且最大程度均勻的 64 種 3-bp gaps，
再由 CorePromoter clean model 掃描最佳 register。

Validation（驗收標準，兩個 anchor 一致）：

- `shift != 0` 就計入 shifted variant。
- m10 shifted rate 與 m35 shifted rate 都必須 `< max_shift_rate`（10%）。
- 所有 shifted variants 都必須在 `-2..+2 bp`；任何 `abs(shift) > 2` 都不通過。
- 只有當下 phase 的 objective 嚴格改善才接受 replacement。

Phase 排程（**驗收線與階段切換點是兩件事**）：

`CONFIG.m10_phase_exit_rate`（預設 20%）只是**切換階段的觸發點**，不是驗收線。搜尋因此是三段式，
而不是把 m10 一路壓到 10% 才碰 -35：

1. **m10 階段** —— m10 shifted rate `> 20%` 時。Objective 依序是
   `(m10 偏移數, m35 偏移數, 超界數, m10 最大絕對偏移, m35 最大絕對偏移)`。
2. **m35 修正階段** —— m10 已 `<= 20%` 而 m35 尚未達標時。Objective 改成
   `(m35 偏移數, m10 偏移數, 超界數, m35 最大絕對偏移, m10 最大絕對偏移)`，
   **m35 排第一**，所以「只降 m10 卻讓 m35 變差」的提案不再獲勝；
   同時加上一條 hard constraint：**提案的 m10 偏移數不得高於當前值（可以相等）**。
   m35 持平但 m10 下降的提案仍算改善並會被接受，免費的 m10 進步不會被丟掉。
3. **回頭補 m10** —— m35 已達標但 m10 還沒到 10% 時，phase 自動回到 m10 繼續壓。

m10 偏移數在階段 1、3 是 objective 首鍵（因此非遞增），在階段 2 被 hard constraint 擋住，
所以整個搜尋期間 **m10 偏移數單調不增**，階段來回不會無限震盪。把
`m10_phase_exit_rate` 設成等於 `max_shift_rate` 就大致回到原本單向的 m10 → m35 流程，
差別只在兩個邊界：閘門用 `<=` 且不看超界，所以 rate 恰好落在門檻上、或已低於門檻但仍有
`|shift| > 2` 時，新版會進 m35 階段，舊版的 `m10_validation_pass` 檢查則不會。


In [ ]:
# === Batch 0: setup and editable design config ===
import importlib
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import automated_promoter_library_design as r
importlib.reload(r)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")


# Shared reporting helpers, used by both Batch 4 (initial) and Batch 7 (final).
def shift_summary_table(summary):
    """Per-anchor view of a summarize_validation() result."""
    return pd.DataFrame([
        {
            "anchor": anchor,
            "shifted_count": summary[f"{anchor}_shifted_count"],
            "shifted_rate": summary[f"{anchor}_shifted_rate"],
            "out_of_range_count": summary[f"{anchor}_out_of_range_count"],
            "max_abs_shift": summary[f"{anchor}_max_abs_shift"],
            "pass": summary[f"{anchor}_validation_pass"],
        }
        for anchor in ("m10", "m35")
    ])


def phase_status_line(summary):
    """One-line view of where the phase schedule currently stands."""
    return (
        f"phase={summary['phase']} | "
        f"m10 gate (<= {CONFIG.m10_phase_exit_rate:.0%}) "
        f"{'open' if summary['m10_phase_gate_pass'] else 'closed'} | "
        f"m10 {summary['m10_shifted_rate']:.2%} pass={summary['m10_validation_pass']} | "
        f"m35 {summary['m35_shifted_rate']:.2%} pass={summary['m35_validation_pass']} | "
        f"global pass={summary['global_validation_pass']}"
    )


def plot_shift_distributions(scan, title_prefix=""):
    """Two-panel m10/m35 shift histogram. English-only labels for VSCode stability."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), dpi=140)
    for ax, anchor in zip(axes, ("m10", "m35")):
        counts = scan[f"{anchor}_shift"].value_counts().sort_index()
        ax.bar(counts.index.astype(int), counts.values, color="#4C72B0")
        ax.set_title(f"{title_prefix}{anchor} shift distribution")
        ax.set_xlabel("Shift (bp)")
        ax.set_ylabel("Variant count")
    plt.tight_layout()
    plt.show()


def plot_element_version_energy_distributions(scan, score_col, title_prefix="", n_bins=60):
    """Six-panel per-element version energy histogram, layered over the whole-library
    baseline. Role mirrors plot_shift_distributions: call once on initial_scan and once
    on result.final_scan (title_prefix="Final ") to compare before/after redesign.
    """
    baseline = scan[score_col].to_numpy(dtype=float)
    edges = np.histogram_bin_edges(baseline, bins=n_bins)
    version_colors = ["#C44E52", "#DD8452", "#55A868", "#4C72B0", "#8172B2"]


    fig, axes = plt.subplots(2, 3, figsize=(16.5, 8.6), dpi=140)
    for ax, element in zip(axes.ravel(), r.ELEMENTS):
        ax.hist(baseline, bins=edges, color="0.75", alpha=0.6, zorder=1,
                 label="All 15,625 variants")

        version_col = f"{element}_version"
        versions = sorted(scan[version_col].unique(), key=r.version_index)
        locked = set(CONFIG.locked_versions(element))
        for version, color in zip(versions, version_colors):
            subset = scan.loc[scan[version_col] == version, score_col].to_numpy(dtype=float)
            label = f"{version} (locked)" if version in locked else version
            ax.hist(subset, bins=edges, histtype="step", linewidth=1.7,
                     color=color, zorder=2, label=label)

        ax.set_title(f"{title_prefix}{element}")
        ax.set_xlabel(f"{score_col} (higher = stronger)")
        ax.set_ylabel("Variant count")
        ax.legend(fontsize=7, loc="upper left")

    fig.suptitle(f"{title_prefix}Full-sequence {score_col} by element version",
                 fontsize=18, y=1.01)
    fig.tight_layout()

    suffix = (title_prefix.strip().lower() or "initial").replace(" ", "_")
    for ext in ("png", "svg"):
        fig.savefig(OUT_DIR / f"element_version_energy_distributions_{suffix}_{score_col}.{ext}",
                    bbox_inches="tight")
    plt.show()


# Select the frozen whole/CorePromoter checkpoint used for full-sequence scanning.
CORE_MODEL_VARIANT = "baseline"  # "baseline" or "tss_pas"
CORE_MODEL_CHECKPOINTS = {
    "baseline": r.WEIGHTS_DIR / "weights_CorePromoter_clean.pt",
    "tss_pas": r.WEIGHTS_DIR / "weights_CorePromoter_tss_pas.pt",
}
CORE_MODEL_CHECKPOINT = CORE_MODEL_CHECKPOINTS[CORE_MODEL_VARIANT]
if not CORE_MODEL_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Missing {CORE_MODEL_VARIANT} checkpoint: {CORE_MODEL_CHECKPOINT}. "
        "Run Model_CorePromoter_TSS_pretrain.ipynb first."
    )

# RUN_MODE = "new": create a new output directory.
# RUN_MODE = "resume": continue an existing directory from its checkpoint/final_elements.
RUN_MODE = "new"  # "new" or "resume"
RESUME_DIR = r.DEFAULT_PARENT_OUT / "automated_redesign_20260716_112038"

if RUN_MODE == "new":
    OUT_DIR = r.DEFAULT_PARENT_OUT / f"automated_redesign_{RUN_STAMP}"
elif RUN_MODE == "resume":
    OUT_DIR = Path(RESUME_DIR)
    if not OUT_DIR.exists():
        raise FileNotFoundError(f"Resume directory does not exist: {OUT_DIR}")
else:
    raise ValueError(f"RUN_MODE must be 'new' or 'resume', not {RUN_MODE!r}")

CACHE_DIR = r.PROJECT_ROOT / "outputs" / "energy_bin_cache"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Edit these sequences when the locked consensus changes.
# One string = one locked slot. A list = several locked slots, weakest first,
# filling the versions above the last mutable bin. UP therefore locks v4 and v5
# and only bins v1-v3; every other element keeps its single locked slot.
CONSENSUS = {
    "UP": ["TGGACTGATATATACAAAA", "CTAATCGGGGGCGATTAAG"],
    "m35": "TTGACA",
    "spacer": "TATGGCGCAAAATGGGG",
    "m10": "TATAAT",
    "DIS": "TTTTATTA",
    "ITS": "CAAAAAAAAG",
}
# "UP": ["TGGACTGATATATACAAAA", "CTAATCGGGGGCGATTAAG"],
# "UP": ["TGGACTGATATATACAAAA", "TAAAAAATTTGGAAAATAG"],

# Per-element pooling boundary, as (lower_fraction, upper_fraction, n_bins) on that
# element's own observed min-max energy axis. Read the fractions straight off the
# Batch 0.5 top axis.
#
# Every bin becomes one mutable version, so N bins give v1..vN plus the locked
# consensus at v{N+1}, and one design state assembles prod(N_e + 1) variants.
# All six at 4 bins -> 5^6 = 15,625, printed below so an edit shows up immediately.
#
# Tune the lower bound to trade weak-sequence coverage against register stability,
# and the upper bound against the locked consensus: v1..vN must all be weaker than
# it, so a bin above the red v5 line in Batch 0.5 has no eligible sequence and
# DesignSpace raises instead of silently picking from a neighbouring bin.
# The spacer needs at least 3 bins because Spacer_v3 = Spacer_v2[:-2] + "TG" is
# derived rather than drawn from bin 3.
ENERGY_BIN_RANGES = {
    # 3 bins, not 4: UP's two locked sequences take v4 and v5, so it still has
    # 5 versions in total and the library stays at 5^6 = 15,625.
    "UP":     (0.0, 0.7, 3),
    "m35":    (0.0, 0.9, 4),
    "spacer": (0.0, 0.9, 4),
    "m10":    (0.0, 0.8, 4),
    "DIS":    (0.0, 1.0, 4),
    "ITS":    (0.0, 1.0, 4),
}

CONFIG = r.DesignConfig(
    consensus=CONSENSUS,
    bg5="CCCTTTCGTCTTCACACAGCAGCAGTCAGGTAGGGAAGAGACC",
    bg3="GTCGACTCTAGA",
    gap_length=3,
    gap_seed=777,
    n_energy_bins=4,  # fallback bin count for any element omitted from the dict below
    mutable_energy_fraction_ranges=ENERGY_BIN_RANGES,
    max_candidates_per_unit=60,
    max_abs_shift=2,
    # Acceptance threshold for BOTH anchors. Unchanged by the phase schedule.
    max_shift_rate=0.10,
    # Phase-switch trigger only. Once m10 is at or below this, the search moves on
    # to the -35 correction phase instead of driving m10 all the way to
    # max_shift_rate first; it returns to m10 after m35 passes. Must be >=
    # max_shift_rate; setting it equal gives back the original single-pass flow
    # except at the threshold itself and while |shift| > 2 variants remain.
    m10_phase_exit_rate=0.20,
    scan_batch_size=15625,
    require_derived_spacer_in_database=False,
)

USE_CACHE = True
MAX_SOURCE_ROWS = None  # None = use all observed sequences in each PKL

SEARCH_SETTINGS = {
    # This is an additional iteration allowance for each new/resume execution.
    "max_iterations": 150,
    "n_driver_units": 3,
    "probe_candidates_per_unit": 2,
    "pair_beam_width": 6,
    "max_pair_evaluations": 8,
    "max_stalled_iterations": 30,
}

config_filename = "run_config.json" if RUN_MODE == "new" else f"resume_config_{RUN_STAMP}.json"
r.save_run_config(
    OUT_DIR, CONFIG, SEARCH_SETTINGS, DEVICE,
    core_model_checkpoint=CORE_MODEL_CHECKPOINT,
    filename=config_filename,
)
print("Run mode:", RUN_MODE)
print("Project root:", r.PROJECT_ROOT)
print("Device:", DEVICE)
print("Output:", OUT_DIR)
print("Core model variant:", CORE_MODEL_VARIANT)
print("Core model checkpoint:", CORE_MODEL_CHECKPOINT)
print()
print("Energy bins and versions per element:")
for element in r.ELEMENTS:
    n_bins, lower, upper, mode = CONFIG.energy_bin_spec(element)
    print(f"  {element:7s} fraction {lower:.2f}-{upper:.2f} into {n_bins} bins"
          f"  ->  mutable {', '.join(CONFIG.mutable_versions(element))}"
          f"  + locked {', '.join(CONFIG.locked_versions(element))}   [{mode}]")
print()
print("Locked sequences (never touched by the search):")
for element in r.ELEMENTS:
    for version, seq in CONFIG.locked_sequences(element).items():
        print(f"  {element:7s} {version}  {seq}")
print("Variants per design state:", f"{CONFIG.n_variants():,}",
      "=", " x ".join(str(len(CONFIG.versions_for(e))) for e in r.ELEMENTS))
print()
print("Phase schedule:")
print(f"  validation (both anchors): shifted rate < {CONFIG.max_shift_rate:.0%}"
      f" and no |shift| > {CONFIG.max_abs_shift} bp")
print(f"  m10 phase   while m10 shifted rate  > {CONFIG.m10_phase_exit_rate:.0%}")
print(f"  m35 phase   once m10 shifted rate  <= {CONFIG.m10_phase_exit_rate:.0%};"
      " m10 shifted count may not rise, m35 leads the objective")
print(f"  back to m10 once m35 passes but m10 is still >= {CONFIG.max_shift_rate:.0%}")


## Batch 0.5: 六個文庫的能量分布與現行 pooling 邊界

在 Batch 1 真正切 bin 之前，先看清楚每個文庫的能量分布長什麼樣，用來決定 Batch 0 的
`ENERGY_BIN_RANGES` 每個元素該填什麼。

每個元素的邊界是 `(lower_fraction, upper_fraction, n_bins)`，**每個 bin 各出一個 mutable
版本**（N 個 bin → `v1..vN` + locked `v{N+1}`）。等寬切法對這些分布並不友善：實測全部六個
文庫都嚴重集中，能量全距的兩端非常稀疏。以 spacer 為例，0–0.8 這段切 4 份的計數是
`32,020 / 246,556 / 53,186 / 1,751` —— 相差兩個數量級，而 0.8–1.0 那段整段只有 52 條；
ITS 的 bin 3 一個人吃掉 511,415 條。這就是為什麼邊界要逐元素看圖決定，而不是統一切全距。

這個 cell 建立的 `element_models` / `scored_pools` / `bin_summary` 會被後面所有 batch 直接
沿用（Batch 1 起不再重建），所以直方圖的橫軸與實際切 bin 的軸是**同一把尺**，不是另外算一套。
畫圖用的 bin 邊界也是呼叫 module 的 `r.pool_bin_edges()`，與 `assign_equal_width_bins`
實際切 bin 用的是同一個 `_bin_edges()`。

每個 panel（順序 UP → -35 → spacer 17 → -10 → DIS → ITS）：

- **橫軸** = 該 element model 算出的 energy（higher = stronger）。**每個 panel 各自縮放**，
  不同 element 的分數不可互相比較。上緣副軸是 `energy_fraction` 0–1，也就是要填進
  `ENERGY_BIN_RANGES` 的那個尺度 —— 可以直接從圖上讀出邊界該設多少。
- **縱軸** = 序列數。計數的是**去重後、長度與 ACGT 都合規**的序列，也就是實際進入
  pooling 的那個 pool 的筆數，不是文庫的 read count。預設 log10 軸，否則稀疏尾巴看不見；
  把 `LOG_COUNTS` 改成 `False` 可看原始線性計數。
- **灰色虛線** = 現行 bin 邊界，上緣標出每個 bin 實際裝到幾條序列。
- **淺藍色區塊** = 目前納入切 bin 的能量範圍；灰底區塊是被 `lower/upper_fraction` 排除掉的
  部分（預設下 UP/spacer/DIS/ITS 是 0.8–1.0，m35/m10 是 0–0.3 與 0.9–1.0）。
- **紅線** = locked 序列的能量，每個 locked 槽位一條。實線是**最弱的那一條**（binding），
  虛線是其餘的。因為 mutable 版本的 hard constraint 是「必須弱於所有 locked」，實線那條才是
  真正決定 `upper_fraction` 能拉到哪的界線：**任何超過實線的 bin 都會抓不到候選而讓 Batch 2
  直接報錯**。表格的 `n_eligible_below_locked` 就是該 bin 扣掉這個限制後真正剩下的條數。
  UP 有兩條紅線（v4 實線、v5 虛線），因為 v5 比 v4 強，binding 的是 v4。

> 幾個一眼可見的陷阱：UP 的紅線在 fraction 0.726，所以 `upper_fraction` 拉到 0.8 以上就開始
> 出現無效 bin（原本 0–1 切 5 份的 bin 5 正是 `n_eligible_below_v5 = 0`）；spacer 的 consensus
> 甚至落在 pool 之外（fraction 1.085，比 SL17 最強的序列還強），m35/m10/DIS 的 consensus 剛好
> 就是 pool 最大值（fraction 1.000）。

-35 / -10 的能量取自 `Model_PL.ipynb` 從 `PL.pkl` 做出來的 BPM 模型
（`BPM/Params_Con17.pkl`，取負號轉成 higher-is-stronger），這正是 pooling 實際用的軸。
`weights_minus35.pt` / `weights_minus10.pt` 雖然在 `weights/` 裡，但 pipeline 沒有任何
code 載入它們（`ElementModelBundle._load_all` 明確跳過），而且與 BPM 排序嚴重不一致
（Spearman ρ = 0.37 / −0.22），所以這裡不使用。

輸出的 `energy_bin_summary.csv` 是整個 run 唯一一份 per-bin 計數表（含 `pct_of_pool`）。


In [ ]:
# === Batch 0.5: per-library energy distributions and current pooling boundaries ===
# English-only plot labels for VSCode/Jupyter rendering stability.
import numpy as np

# log10 y axis. The spacer pool puts 246,556 sequences in bin 2 and only 52 in
# bin 5, so a linear axis hides exactly the sparse tails the boundary decision
# depends on. Set False to read raw linear counts.
LOG_COUNTS = True
SHOW_FRACTION_AXIS = True   # top axis = energy_fraction, the scale that
                            # CONFIG.mutable_energy_fraction_ranges is written in
N_HIST_BINS = 80

PANEL_LABELS = {"UP": "UP", "m35": "-35", "spacer": "spacer 17",
                "m10": "-10", "DIS": "DIS", "ITS": "ITS"}
# -35/-10 come from the BPM model that Model_PL.ipynb derives from PL.pkl
# (BPM/Params_Con17.pkl), negated to a higher-is-stronger axis. The orphaned
# weights_minus35.pt / weights_minus10.pt are deliberately not used here:
# nothing in the pipeline loads them and they disagree with BPM
# (Spearman rho = 0.37 / -0.22), so they cannot explain the bins below.
PANEL_NOTES = {
    "UP": "UL.pkl, 19 bp, NN",
    "m35": "PL.pkl:minus35, 6 bp, BPM -dG",
    "spacer": "SL17.pkl, 17 bp, NN",
    "m10": "PL.pkl:minus10, 6 bp, BPM -dG",
    "DIS": "DL.pkl, 8 bp, NN",
    "ITS": "ITS.pkl, 10 bp, NN",
}

# Built once here and reused by every later batch, so the histogram and the bins
# it diagnoses share one score axis. build_scored_pools caches on file signatures.
element_models = r.ElementModelBundle(DEVICE)
scored_pools = r.build_scored_pools(
    models=element_models,
    config=CONFIG,
    cache_dir=CACHE_DIR,
    use_cache=USE_CACHE,
    max_source_rows=MAX_SOURCE_ROWS,
)
bin_summary = r.energy_bin_summary(scored_pools, CONFIG, element_models)

stats_rows = []
fig, axes = plt.subplots(2, 3, figsize=(16.5, 8.6), dpi=140)

for ax, element in zip(axes.ravel(), r.ELEMENTS):
    pool = scored_pools[element]
    energy = pool["energy"].to_numpy(dtype=float)
    edges, e_min, e_max, span, frac_lo, frac_hi, n_bins = r.pool_bin_edges(pool)
    # "v4=1.1085;v5=1.3915" -> {"v4": 1.1085, "v5": 1.3915}
    locked_spec = bin_summary.loc[bin_summary["element"] == element, "locked_energies"].iloc[0]
    locked_e = {k: float(v) for k, v in (t.split("=") for t in str(locked_spec).split(";"))}
    # Mutable candidates must clear the weakest locked slot, so that is the line
    # the bin edges actually have to stay under.
    bind_v = min(locked_e, key=locked_e.get)
    v_bind = locked_e[bind_v]

    # lower_fraction > 0 leaves out-of-range sequences with <NA> energy_bin.
    bin_counts = (
        pool["energy_bin"].dropna().astype(int).value_counts()
        .reindex(range(1, n_bins + 1), fill_value=0).astype(int)
    )

    if frac_lo > 0.0:
        ax.axvspan(e_min, edges[0], color="0.82", alpha=0.6, zorder=0,
                   label="Outside binned range")
    if frac_hi < 1.0:
        ax.axvspan(edges[-1], e_max, color="0.82", alpha=0.6, zorder=0,
                   label=None if frac_lo > 0.0 else "Outside binned range")
    ax.axvspan(edges[0], edges[-1], color="#4C72B0", alpha=0.07, zorder=0,
               label="Binned range")

    ax.hist(energy, bins=N_HIST_BINS, color="#4C72B0", zorder=2)
    for i, edge in enumerate(edges):
        ax.axvline(edge, color="0.30", ls="--", lw=0.9, zorder=3,
                   label="Current bin edge" if i == 0 else None)
    for i, (lv, le) in enumerate(sorted(locked_e.items(), key=lambda kv: kv[1])):
        ax.axvline(le, color="#C44E52", lw=1.8 if lv == bind_v else 1.1,
                   ls="-" if lv == bind_v else ":", zorder=4,
                   label="Binding locked slot" if i == 0 else
                         ("Other locked slot" if i == 1 else None))
        ax.text(le, 0.98, lv, transform=ax.get_xaxis_transform(), ha="right",
                va="top", fontsize=8, color="#C44E52", rotation=90)

    # Headroom above the tallest histogram bar for the per-bin count labels.
    top = max(int(np.histogram(energy, bins=N_HIST_BINS)[0].max()), 1)
    if LOG_COUNTS:
        ax.set_yscale("log")
        ax.set_ylim(0.7, top * 10 ** 0.60)
        label_y = top * 10 ** 0.14
    else:
        ax.set_ylim(0, top * 1.32)
        label_y = top * 1.14
    for bin_id, lo, hi in zip(range(1, n_bins + 1), edges[:-1], edges[1:]):
        # The white backing keeps the v5 line from cutting through the number.
        ax.text((lo + hi) / 2.0, label_y, f"bin {bin_id}\n{bin_counts[bin_id]:,}",
                ha="center", va="bottom", fontsize=9, color="0.25", zorder=5,
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.75))

    # A locked sequence can sit outside the observed pool range - the spacer
    # consensus scores 4.66 while the strongest sequence in SL17.pkl only reaches
    # 4.05 - so widen the view to keep those lines visible instead of clipping.
    view_lo = min([e_min, *locked_e.values()])
    view_hi = max([e_max, *locked_e.values()])
    pad = 0.02 * (view_hi - view_lo)
    ax.set_xlim(view_lo - pad, view_hi + pad)
    ax.set_title(
        f"{PANEL_LABELS[element]}   ({PANEL_NOTES[element]})\n"
        f"n = {len(pool):,}   |   {n_bins} bins over fraction {frac_lo:.2f}-{frac_hi:.2f}",
        fontsize=12, pad=20 if SHOW_FRACTION_AXIS else 6,
    )
    if SHOW_FRACTION_AXIS:
        sec = ax.secondary_xaxis(
            "top",
            functions=(lambda e, o=e_min, s=span: (e - o) / s,
                       lambda f, o=e_min, s=span: o + f * s),
        )
        sec.tick_params(labelsize=12, pad=1)

    fraction = (energy - e_min) / span
    inside = (fraction >= frac_lo - 1e-12) & (fraction <= frac_hi + 1e-12)
    p1, p25, p50, p75, p99 = np.percentile(energy, [1, 25, 50, 75, 99])
    stats_rows.append({
        "element": element,
        "source": PANEL_NOTES[element],
        "n_sequences": int(len(energy)),
        "energy_min": e_min, "p1": p1, "p25": p25, "median": p50,
        "p75": p75, "p99": p99, "energy_max": e_max,
        "binning_mode": str(pool["binning_mode"].iloc[0]),
        "n_bins": n_bins,
        "fraction_lower": frac_lo, "fraction_upper": frac_hi,
        "range_energy_lower": float(edges[0]), "range_energy_upper": float(edges[-1]),
        "n_inside_range": int(inside.sum()),
        "frac_inside_range": float(inside.mean()),
        "locked_versions": ";".join(f"{k}={v:.4f}" for k, v in locked_e.items()),
        "binding_locked_version": bind_v,
        "binding_locked_energy": v_bind,
        "binding_locked_fraction": float((v_bind - e_min) / span),
        "n_below_binding_locked": int((energy < v_bind).sum()),
    })

handles, labels = [], []
for ax in axes.ravel():
    for handle, label in zip(*ax.get_legend_handles_labels()):
        if label not in labels:
            handles.append(handle)
            labels.append(label)

fig.suptitle("Per-library energy distributions and current pooling boundaries",
             fontsize=24, y=0.995)
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.945),
           ncol=len(labels), frameon=False, fontsize=18)
fig.supxlabel("Element model energy, higher = stronger "
              "(per-panel scale; not comparable across elements). "
              "Top axis = energy_fraction 0-1.", fontsize=14)
fig.supylabel(f"Sequence count{' (log10)' if LOG_COUNTS else ''}", fontsize=16)
fig.tight_layout(rect=(0.012, 0.015, 1, 0.905))
for ext in ("png", "svg"):
    fig.savefig(OUT_DIR / f"element_energy_distributions.{ext}", bbox_inches="tight")
plt.show()

element_energy_stats = pd.DataFrame(stats_rows)
# The run's only per-bin count table; Batch 1 no longer writes a second copy.
pool_sizes = {element: len(scored_pools[element]) for element in r.ELEMENTS}
bin_summary["pct_of_pool"] = (
    100.0 * bin_summary["n_sequences"] / bin_summary["element"].map(pool_sizes)
)

element_energy_stats.to_csv(OUT_DIR / "element_energy_distribution_stats.csv", index=False)
bin_summary.to_csv(OUT_DIR / "energy_bin_summary.csv", index=False)
display(element_energy_stats.round(4))
display(bin_summary.round(4))
print("Figure and tables written to:", OUT_DIR)


## Batch 1: load the full-sequence CorePromoter model

`element_models` / `scored_pools` / `bin_summary` 已經在 Batch 0.5 建好並直接沿用，這裡只補上
掃描整條序列用的 CorePromoter clean model —— 之前這個 cell 會把 element models 與 scored pools
再建一次（重新從硬碟載 5 個 NN checkpoint），那是純粹的重複。

UP/Spacer/DIS/ITS 使用各自已訓練的 element weights。`Model_PL.ipynb` 的 -35/-10 data flow 使用 BPM，因此這兩個元素使用 `-BPM dG` 作為 higher-is-stronger score。不同 element 的分數不可互相比較。

切幾個 bin、切在哪一段，完全由 Batch 0 的 `ENERGY_BIN_RANGES` 決定（見 Batch 0.5 的分布圖）。
`energy_bin_summary` 逐 bin 列出 `n_sequences` 與 `n_eligible_below_locked` —— 後者才是真正可用的
候選數，因為 mutable 版本必須弱於所有 locked 序列（比較對象是 `binding_locked_version` 那一條）；
某個 bin 的 `n_eligible_below_locked` 為 0 時，Batch 2 會直接報錯而不是跨 bin 補候選。

Scored pools 會依 PKL 與 weight/BPM parameter 的檔案 signature 快取；來源或模型更新後會自動重建。改動 bin 邊界不需要重算能量，快取仍然有效。


In [ ]:
core_model = r.load_core_model(DEVICE, checkpoint_path=CORE_MODEL_CHECKPOINT)

print("Core model:", CORE_MODEL_CHECKPOINT)
print("Scored pool sizes:", {e: f"{len(scored_pools[e]):,}" for e in r.ELEMENTS})


## Batch 2: construct database-backed design units

這個步驟執行 hard constraints：

- 一般 element 的 `v1..vN` 必須各自來自對應的 bin 1..N（N = 該元素的 `n_bins`）。
- 所有 mutable 版本的 score 必須低於**每一條** locked 序列，也就是低於 `binding_locked_version`
  （最弱的那一條）。UP 的比較對象是 v4 而不是 v5。
- locked 序列本身逐條核對，被改動就報錯。
- 同一 element 的各條 sequence 不可重複。
- Spacer_v2/v3 必須滿足 coupled rule（v3 = v2[:-2] + "TG"，不取自 bin 3）。
- 任一必要 bin 沒有 eligible sequence 時直接報錯，不跨 bin 補候選 —— 這通常表示該 bin 的上界
  設得比 locked consensus 還強，回 Batch 0 把 `upper_fraction` 調低。


In [ ]:
design_space = r.DesignSpace(
    config=CONFIG,
    models=element_models,
    scored_pools=scored_pools,
)

if RUN_MODE == "resume":
    saved_elements_path = OUT_DIR / "current_elements_checkpoint.csv"
    if not saved_elements_path.exists():
        saved_elements_path = OUT_DIR / "final_elements.csv"
    if not saved_elements_path.exists():
        raise FileNotFoundError(f"No resume elements found in {OUT_DIR}")
    initial_elements = pd.read_csv(saved_elements_path)
    initial_state = design_space.state_from_selected_elements(initial_elements)
    print("Recovered state from:", saved_elements_path)
else:
    initial_state = design_space.initial_state()
    initial_elements = design_space.selected_elements(initial_state)

unit_summary = design_space.candidate_pool_summary()

# initial_elements.csv is written by AutomatedRedesigner.run() in Batch 6.
unit_summary.to_csv(OUT_DIR / "design_unit_candidate_counts.csv", index=False)
display(initial_elements)
display(unit_summary)


## Batch 3: fixed balanced 3-bp gap assignment and library assembly

Variant 數是 `∏(N_e + 1)`，由 Batch 0 的 bin 數決定；六個元素都是 4 個 bin 時為 15,625。

64 種 3-bp gap 要盡量平均分配到這些 variants 上。`15,625 = 64 × 244 + 9`，無法完全等量，
所以最大程度均勻的固定分配是 55 種 gap 各 244 次、9 種各 245 次（`build_balanced_gap_assignment`
會斷言任兩種 gap 的次數差不超過 1，因此改變 bin 數之後這個分配仍然成立，只是餘數不同）。
這份 assignment 在所有 replacement 前後保持不變。


In [ ]:
saved_gap_path = OUT_DIR / "gap_assignment.csv"
if RUN_MODE == "resume" and saved_gap_path.exists():
    gap_assignment = pd.read_csv(saved_gap_path)
    print("Loaded fixed gap assignment:", saved_gap_path)
else:
    gap_assignment = r.build_balanced_gap_assignment(CONFIG)

initial_variants = r.assemble_library(initial_elements, gap_assignment, CONFIG)

gap_counts = (
    gap_assignment["gap_3bp"]
    .value_counts()
    .rename_axis("gap_3bp")
    .reset_index(name="n_variants")
    .sort_values("gap_3bp")
)
# gap_assignment.csv is written by AutomatedRedesigner.run() in Batch 6. The
# assignment is deterministic in gap_seed, so an interrupted run regenerates the
# identical table on resume.
if RUN_MODE == "new":
    initial_variants.to_csv(OUT_DIR / f"initial_assembled_{CONFIG.n_variants()}.csv", index=False)

print("Assembled variants:", f"{len(initial_variants):,}")
print("Gap count distribution:", gap_counts["n_variants"].value_counts().sort_index().to_dict())
display(gap_counts)


## Batch 4: initial CorePromoter register scan

正式欄位：

```text
m10_shift = observed_m10_start - design_m10_start
m35_shift = observed_m35_start - design_m35_start
spacer_length_shift = observed_spacer_len - design_spacer_len
```

Phase 由 `summarize_validation` 決定：m10 shifted rate 高於 `CONFIG.m10_phase_exit_rate`（20%）
時用 `m10_shift`；一旦 m10 降到 20% 以內就切到 `m35_shift`；m35 達標而 m10 仍未低於
`max_shift_rate`（10%）時再切回 `m10_shift`。`m10_phase_gate_pass` 這個欄位就是那道 20% 的閘門，
與 `m10_validation_pass`（10% 驗收）是兩個不同的旗標。

掃描同時輸出八個 architecture windows、observed element-role sequences、落在哪些 design regions，以及同一 element model 內的 delta energy。


In [ ]:
scanner = r.CorePromoterScanner(
    model=core_model,
    element_models=element_models,
    config=CONFIG,
    device=DEVICE,
)

initial_scan = scanner.scan(initial_variants, annotate_element_energies=True)
initial_summary = r.summarize_validation(initial_scan, CONFIG)
# initial_scan_{n}.csv is written by AutomatedRedesigner.run() in Batch 6.

display(shift_summary_table(initial_summary))
print(phase_status_line(initial_summary))


In [ ]:
plot_shift_distributions(initial_scan)


In [ ]:
plot_element_version_energy_distributions(initial_scan, "design_core_score")
plot_element_version_energy_distributions(initial_scan, "best_core_score")


## Batch 5: dominant shift and conditional-risk diagnosis

診斷的對象是**當前 phase 那一欄**（`m10_shift` 或 `m35_shift`）所有 `shift != 0` 的 variants；
先找數量最多的 shift coordinate（`_dominant_shift`，超界只當作次要 tie-breaker），
再以 `P(dominant shift | element version)` 排名 driver。

Observed role 落在某個 design region 只用來提供 redesign evidence；不同 element model 的 raw delta energy不互相比較。最終 replacement 仍由完整 15,625 before/after validation 決定。


In [ ]:
initial_diagnosis = r.diagnose_shift_drivers(
    scan=initial_scan,
    selected_elements=initial_elements,
    design_space=design_space,
    config=CONFIG,
)

print("Optimization phase:", initial_diagnosis["phase"])
print("Dominant shift coordinate:", initial_diagnosis["dominant_shift"])
display(initial_diagnosis["risk"].head(20))
display(initial_diagnosis["overlap"].head(30))


## Batch 6: monotonic automated redesign with live progress and resume

每輪會 print iteration、phase（含 20% 閘門是 open 還是 closed）、目前 m10/m35 shifted rate、out-of-range count、dominant shift、driver units、每個 proposal 結果與接受/拒絕原因。

接受規則依 phase 而不同：

- **m10 階段**：objective `(m10 偏移數, m35 偏移數, 超界數, …)` 嚴格下降。
- **m35 階段**：先用 hard filter 剔除**任何讓 m10 偏移數上升**的 proposal（持平可以），
  再要求 objective `(m35 偏移數, m10 偏移數, 超界數, …)` 嚴格下降。

`search_progress.csv` 每輪多一欄 `m10_phase_gate_pass`，可以直接看出哪一輪跨過 20% 閘門切進 -35 階段、哪一輪又切回 m10。

執行期間會在 `OUT_DIR` 持續覆寫：

- `search_progress.csv`：每輪狀態的持續更新 DataFrame。
- `proposal_history_checkpoint.csv`：所有已測 single/double proposals。
- `current_elements_checkpoint.csv`：目前接受的 30 條 sequences。
- `current_validation.json`：目前 validation。
- `search_checkpoint.json`：state、最後完成輪數、stalled counter 與已測 proposals。

`progress_df` 也會在記憶體中每輪原地更新；手動 interrupt 後可直接 `display(progress_df.tail())`。

若中途停止 kernel，將 setup cell 的 `RUN_MODE` 改為 `"resume"`，並把 `RESUME_DIR` 指向原本的 output directory；Batch 6 會從下一個 iteration 繼續。每次 resume 的 `max_iterations` 是額外輪數，不是總累積上限。


In [ ]:
redesigner = r.AutomatedRedesigner(
    design_space=design_space,
    scanner=scanner,
    gap_assignment=gap_assignment,
    config=CONFIG,
    out_dir=OUT_DIR,
    **SEARCH_SETTINGS,
)

# Mutated in place after every completed iteration. It remains available if
# this cell is manually interrupted.
progress_df = pd.DataFrame()

result = redesigner.run(
    initial_state=initial_state,
    initial_evaluation=(initial_elements, initial_scan, initial_summary),
    resume=(RUN_MODE == "resume"),
    reset_stalled_on_resume=True,
    progress_df=progress_df,
    verbose=False,
)

print("Success:", result.success)
print("Stop reason:", result.stop_reason)
print("Saved to:", result.out_dir)
display(pd.DataFrame([result.final_summary]))
display(progress_df)


## Batch 7: final design and audit tables

`final_elements.csv` 是最後保留的 30 條 element sequences；`proposal_history.csv` 同時保留 accepted 與 rejected moves，方便追蹤為何某次替換沒有被採用。


In [ ]:
display(result.final_elements)
display(result.final_risk.head(20))
display(result.final_overlap.head(30))

display(shift_summary_table(result.final_summary))
print(phase_status_line(result.final_summary))
plot_shift_distributions(result.final_scan, title_prefix="Final ")

accepted = result.proposals[result.proposals["accepted"]] if len(result.proposals) else result.proposals
display(accepted)


In [ ]:
plot_element_version_energy_distributions(result.final_scan, "design_core_score", title_prefix="Final ")
plot_element_version_energy_distributions(result.final_scan, "best_core_score", title_prefix="Final ")
